# D1：PyBaMM 电池仿真 —— 充放电特性
# D1: PyBaMM Battery Simulation — Charge/Discharge

## 一、实验原理 | Principle

### 1.1 为什么仿真电池？
电池是电动车最贵的部件。设计电池包前，需要预测：
- 不同倍率放电下，电压掉多快？（影响动力）
- 能放出多少容量？（影响续航）
- 快充会不会析锂？（影响寿命和安全）

### 1.2 PyBaMM
**PyBaMM**（剑桥、牛津等开发）：基于**电化学偏微分方程**的电池仿真，
不是简单的等效电路拟合。**SPM**（单颗粒模型）把每个电极简化为一个球形颗粒，
计算快、适合教学。**Chen2020** 是商用 NMC 电池的实测参数。

### 1.3 关键概念
- **C-rate**：充放电倍率。1C = 1小时放完，2C = 半小时放完
- **OCV**：开路电压（静置时的电压）；**SOC**：剩余电量百分比
- **极化**：大电流放电时电压的额外下降（内阻效应）

## 二、实验目的 | Objectives
1. 掌握 PyBaMM 的 SPM 模型和 Chen2020 参数集
2. 仿真 1C 放电，观察电压/容量曲线
3. 导出 OCV-SOC 数据（供 C1 的 FASTSim 用——这就是 D→C 的链路！）

## 三、学生任务 | Your Tasks
1. **运行** 1C 放电仿真，记录放出的容量（Ah）
2. **对比**：0.5C 和 2C 放电，容量有什么变化？为什么？（提示：极化）
3. **导出**：用 `BatteryFASTSimAdapter.export_from_pybamm()` 生成 CSV
4. **思考题**：快充（3C）时，负极表面会发生什么？（提示：析锂，查 D4）

## 四、验证内容 | Verification
- [ ] PyBaMM 仿真收敛（无求解器报错）
- [ ] 放电容量误差 ≤5%（vs 标称 4.85Ah）
- [ ] 成功导出 OCV-SOC CSV 文件
- [ ] CSV 可被 `BatteryFASTSimAdapter` 正确读取

### 常见报错说明 | Troubleshooting
| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `Solver failed` | 求解器不收敛 | 降低 C-rate，或换 SPM 模型 |
| 容量远小于标称 | 参数或倍率设置错误 | 检查 `ParameterValues` 是否为 Chen2020 |
| `KeyError: 'Terminal voltage'` | 变量名错误 | PyBaMM 变量名区分大小写，查文档 |


In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
import pybamm
pybamm.set_logging_level('ERROR')
from common.battery_adapter import BatteryFASTSimAdapter
import matplotlib.pyplot as plt
import numpy as np
print('✓ 导入成功')

In [ ]:
# 1C 放电
model = pybamm.lithium_ion.SPM()
param = pybamm.ParameterValues('Chen2020')
sim = pybamm.Simulation(model, parameter_values=param)
sim.solve([0, 3600])
sol = sim.solution

t = sol['Time [s]'].entries
V = sol['Terminal voltage [V]'].entries
cap = sol['Discharge capacity [A.h]'].entries

plt.figure(figsize=(10, 4))
plt.plot(cap, V)
plt.xlabel('放出容量 (Ah)'); plt.ylabel('端电压 (V)')
plt.title('1C 放电曲线'); plt.grid(True, alpha=0.3)
plt.show()

nominal_cap = cap[-1]
print(f'放出容量: {nominal_cap:.2f} Ah')

# 断言：容量合理（Chen2020 约 4.85Ah）
assert abs(nominal_cap - 4.85)/4.85 <= 0.05, '容量异常!'
print('✅ 评测通过！')

In [ ]:
# 导出 OCV-SOC CSV（演示用简化数据）
soc = np.linspace(0, 1, 20)
# 用放电曲线的电压近似 OCV（教学简化）
ocv = np.interp(soc, cap/cap[-1], V)

BatteryFASTSimAdapter.export_from_pybamm(
    soc, ocv, '/tmp/battery_pack_NMC_75Ah.csv',
    chem='NMC', capacity_ah=75.0)
print('✓ CSV 已导出: /tmp/battery_pack_NMC_75Ah.csv')

# 验证 Adapter 可加载
adapter = BatteryFASTSimAdapter('/tmp/battery_pack_NMC_75Ah.csv')
e_kwh = adapter.equivalent_energy_kwh(75.0)
print(f'等效能量: {e_kwh:.1f} kWh')

## 🏆 挑战任务
1. 对比 0.5C/1C/2C 放电曲线，解释倍率效应。
2. 把导出的 CSV 用 `adapter.apply_to_vehicle()` 映射到 FASTSim 车型，跑 UDDS 看电耗变化。